Environment Setup & Google Drive Mount



In [ ]:
# Install ASE, MACE, and hardware acceleration bindings
!pip install ase mace-torch
!pip install cuequivariance cuequivariance-torch cuequivariance-ops-torch-cu12

from google.colab import drive
import os

# Force a remount to refresh the Colab file system cache
drive.mount('/content/drive', force_remount=True)

# The path points to the shortcut you just placed in My Drive
shared_base_path = '/content/drive/MyDrive/3.POD'

if os.path.exists(shared_base_path):
    print("Success! Available directories:")
    print(os.listdir(shared_base_path))
else:
    print(f"Error: Colab still cannot find {shared_base_path}. Check that the shortcut is exactly named '3.POD' and sits directly in My Drive.")

Data Aggregation & Training (From-Scratch)

In [ ]:
import os
import gc
import sys
import torch
import random
from ase.io import read, write
from mace.cli.run_train import main as mace_train_main

# =============================================================================
# CONFIGURATION FOR OPTION A: ARCHITECTURE COMPARISON
# =============================================================================
# Run this script 3 separate times, changing these two variables each time:
# Run 1 (Small):  MODEL_NAME = "scratch_L0", L_MAX = 0
# Run 2 (Medium): MODEL_NAME = "scratch_L1", L_MAX = 1
# Run 3 (Large):  MODEL_NAME = "scratch_L2", L_MAX = 2

MODEL_NAME = "scratch_L2"
L_MAX = 2

# =============================================================================
# ENVIRONMENT & PATH SETUP
# =============================================================================
gc.collect()
torch.cuda.empty_cache()

base_dir = '/content/drive/MyDrive/3.POD'
output_dir = '/content/drive/MyDrive/MDProject/universal_model'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'results'), exist_ok=True)

combined_train_path = os.path.join(output_dir, 'train_combined.extxyz')
combined_valid_path = os.path.join(output_dir, 'valid_combined.extxyz')

# =============================================================================
# UNIVERSAL DATA AGGREGATION
# =============================================================================
print("--- STARTING DATA AGGREGATION ---")
all_frames = []

for folder_name in os.listdir(base_dir):
    if 'NVT' in folder_name or 'NPT' in folder_name:
        state_path = os.path.join(base_dir, folder_name)
        if not os.path.isdir(state_path): continue

        for root, dirs, files in os.walk(state_path):
            target_file = None
            if 'cleaned_dataset.extxyz' in files:
                target_file = os.path.join(root, 'cleaned_dataset.extxyz')
            elif 'train.extxyz' in files:
                target_file = os.path.join(root, 'train.extxyz')
            elif 'OUTCAR' in files:
                target_file = os.path.join(root, 'OUTCAR')

            if target_file and os.path.getsize(target_file) > 1024:
                try:
                    frames = read(target_file, index='-50:')
                    all_frames.extend(frames)
                except Exception as e:
                    pass

random.seed(42)
random.shuffle(all_frames)
split_idx = int(len(all_frames) * 0.9)

write(combined_train_path, all_frames[:split_idx], format='extxyz')
write(combined_valid_path, all_frames[split_idx:], format='extxyz')

# =============================================================================
# MACE TRAINING (FROM SCRATCH)
# =============================================================================
print(f"\n--- STARTING MACE TRAINING FOR {MODEL_NAME} ---")

custom_args = [
    "mace_run_training",
    f"--name={MODEL_NAME}",
    f"--train_file={combined_train_path}",
    f"--valid_file={combined_valid_path}",
    f"--checkpoints_dir={output_dir}/checkpoints",
    f"--results_dir={output_dir}/results",
    "--energy_key=energy",
    "--forces_key=forces",
    "--stress_key=stress",
    "--compute_stress=True",
    "--stress_weight=1.0",
    "--energy_weight=1.0",
    "--forces_weight=100.0",

    # Initialization changed from Foundation to Average for training from scratch
    "--E0s=average",
    f"--max_L={L_MAX}",

    "--r_max=5.0",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--max_num_epochs=50",
    "--patience=10",
    "--lr=0.005",                   # Increased learning rate slightly for from-scratch training
    "--ema",
    "--swa",
    "--start_swa=30",
    "--scaling=rms_forces_scaling",
    "--atomic_numbers=[1, 6, 7, 50, 53]",
    "--device=cuda",
    "--default_dtype=float32",
    "--enable_cueq=False",
    "--restart_latest"              # Hardware acceleration enabled safely
]

sys.argv = custom_args
mace_train_main()
print(f"\nTraining for {MODEL_NAME} successfully completed!")

Fine-Tuning the Foundation Model

In [ ]:
import os
import gc
import sys
import torch
import random
from ase.io import read, write
from mace.cli.run_train import main as mace_train_main

# =============================================================================
# THE ADAPTED EXPERT: FINE-TUNING CONFIGURATION
# =============================================================================
MODEL_NAME = "finetuned_L0"
FOUNDATION_SIZE = "small"
L_MAX = 0

gc.collect()
torch.cuda.empty_cache()

base_dir = '/content/drive/MyDrive/3.POD'
output_dir = '/content/drive/MyDrive/MDProject/universal_model'

combined_train_path = os.path.join(output_dir, 'train_combined.extxyz')
combined_valid_path = os.path.join(output_dir, 'valid_combined.extxyz')

# =============================================================================
# MACE FINE-TUNING
# =============================================================================
print(f"\n--- STARTING MACE FINE-TUNING FOR {MODEL_NAME} ---")

custom_args = [
    "mace_run_training",
    f"--name={MODEL_NAME}",
    f"--train_file={combined_train_path}",
    f"--valid_file={combined_valid_path}",
    f"--checkpoints_dir={output_dir}/checkpoints",
    f"--results_dir={output_dir}/results",
    "--energy_key=energy",
    "--forces_key=forces",
    "--stress_key=stress",
    "--compute_stress=True",
    "--stress_weight=1.0",
    "--energy_weight=1.0",
    "--forces_weight=100.0",

    # --- FOUNDATION MODEL ARGUMENTS RE-ACTIVATED ---
    "--E0s=foundation",
    f"--foundation_model={FOUNDATION_SIZE}",
    "--multiheads_finetuning=True",          # Prevents catastrophic forgetting
    "--pt_train_file=mp",                    # Downloads Materials Project data
    "--num_samples_pt=1000",                 # Mixes 1000 MP frames with your data

    f"--max_L={L_MAX}",
    "--r_max=5.0",
    "--batch_size=5",
    "--valid_batch_size=5",
    "--max_num_epochs=50",
    "--patience=10",
    "--lr=0.0001",                           # Lower learning rate required for fine-tuning
    "--ema",
    "--swa",
    "--start_swa=30",
    "--scaling=rms_forces_scaling",
    "--atomic_numbers=[1, 6, 7, 50, 53]",
    "--device=cuda",
    "--default_dtype=float32",

    # --- BUG AVOIDANCE ---
    "--enable_cueq=False"                    # Disabled to allow e3nn weights to load safely
]

sys.argv = custom_args
mace_train_main()
print(f"\nFine-Tuning for {MODEL_NAME} successfully completed!")